# <center> OriGENE  <center>
    
## <center> *Cancer gene classification using deep convolutional neural networks on epigenetic marker enrichment profiles* <center> 
    
**Authors:**  *Marc Pielies-Avellí*, *Ming-Heng Hsiung*  *Dr. Alin S. Tomoiaga*, *Dr. Mattias Ohlsson*, *Dr. Victor Olariu*
    
**Links of interest:** 
    
Genome Browser: <https://genome.ucsc.edu/>
    
Cancer Gene Census: gold standard, curated database. <https://cancer.sanger.ac.uk/census>
  
Raw data: *Chen K et al.* 2015  <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE67471>

ENCODE project data for NT2-D1: <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE31755>


 
**Main References:**
- Jie Lyu et al., 2020: <https://www.science.org/doi/10.1126/sciadv.aba6784>
    
- Chen K et al., 2015:  <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE67471>

- Sherman M.A. et al., 2022: <https://doi.org/10.1038/s41587-022-01353-8>
    

**ABSTRACT:** *In this project we will study how post-transcriptional modifications in the lysines at the tail of the histones (proteins around which DNA is coiled) can be used to predict whether a certain gene can behave as an oncogene/tumor supressor or not. We will use a DNN to extract features from epigenetic sequences (H3K4me3 for cancer and healthy matching tissues, Multiple markers from single patient) and train it.* 

Key words:  Tumor supressor gene (TSG), Oncogene (OG), Cancer Driver (CD), Normal Gene (NG), epigenetic marker, H3K4me3, H3K4me1, DNN

## <center> TABLE OF CONTENTS <center>

#### A) SECTION I: Raw data obtention...............................................................................................................................................
1.1 SUMMARY: From raw data to the model outputs
    
1.2 Read ChIP-seq output txt files and create target vector
    
1.3 Storing the position values, signal values and targets in arrays
    
1.4 PLOTTING the entire raw signals, with varying length for each gene

#### B) SECTION II: Data preprocessing ............................................................................................................................................
2.1 Data preprocessing
    
2.2 Final plots
    
2.3 ESTIMATING THE PERCENTAGE OF NOISY SEQUENCES
    
2.4 Clustering

#### C) SECTION III: Training and predicting  ................................................................................................................................................................
3.1 Training
    
3.2 Auxilliary visualization functions
    
3.3 THE MODELS
    
3.4 Training and K-fold cross testing of the models




    

# <center> SECTION I: RAW DATA OBTENTION <center> 



    
### <center> 1.1 SUMMARY: From raw data to the model outputs  <center>
    
<center> ____________________________________________________________________________<center>
    
    
 **1)  Retrieve the data:**
 
 The files that we are going to work with occupy GBs, so we need to find a directory with enough space in the disk to allocate them. In order to ease the reproducibility of this section, a Snakemake workflow was written. However, here I will also give an overview of the process.
 
The raw SRA files for the the epigenetic markers (H3K4me3,H3K4me1, H3K27ac...) can be downloaded from the GEO webpage (e.g.  https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE26320) and we can look for them using their file accession numbers (GSE...) and sample numbers (GSM...). Once we are in the GSM... page, go to Relations -> SRA and Click.
Then click on the Run SRA link, SRR..., and go to Data access, where we'll find the full path of the files to download. (e.e. https://trace.ncbi.nlm.nih.gov/Traces/sra/?run=SRR094390)

A) To get the files in a linux machine use (I use the NCBI link): 

``` wget https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos1/sra-pub-run-1/SRR094390/SRR094390.2 ```

B) To dump the SRA files to a .fastq file:

```fastq-dump SRR094310.2```


C) To map the file to hg38, the latest standarized assembly of the human genome, and get a sam file with all the alignments:

``` bowtie2 -x /export/scratch/marc/marc/marc_directory/bowtie2-2.4.4-linux-x86_64/hg38 -p 4 SRR094310.2.fastq -S SRR094310.2.sam ```


D) convert SAM to binary equivalent BAM:

``` samtools view -S -b SRR6511166.1.sam > SRR6511166.1.bam ```

E) Sort BAM:

```samtools sort result.bam -o result.bam.sorted ```

F) Remove duplicated reads (PCR origin mostly)

```samtools rmdup [-sS] input.srt.bam out.bam ```


G) Sorted bam to bedgraph:

``` bedtools genomecov -bg -ibam result.bam.sorted > result.bedGraph ```


H) Sorting the bedgraph:

``` bedSort SRR6511166.1.bedGraph SRR6511166.1.bedGraph.sorted ```

The SRR6511166.1.bedGraph.sorted resulting file could look like:
    
chr1    10000   10007   1
    
chr1    10007   10029   2
    
chr1    10029   10030   3
    
chr1    10030   10031   4
    
Where we have the name of the chromosome in the first column, the first and last position of a bin (in basepairs, from left to right in the genome for the positive strand) in second and third columns, and finally the number of reads that were found in this region.
    
    
    

 **2)  Create the chromosome files:**
    
The easiest, fastest and most efficient way to split the previous file into chromosomes is by running the following command from the output directory, e.g.:
    
*'/scratch/lema/marc/oncogenes/GSE67471/chromosomes/GSM1647618_chr/'*
    
The command is:
 
    ```awk  '{print > "{SRA file title}_"$1".txt"}' .../{SRA file title}.bedgraph.sorted ```
    
    
    
 
 
 **3) Create the gene files:** 
 
 Once we have the chromosome files, we can create the txt files for each gene using  
 
 ``` create_txt(Diagnostic_file,PATH,outPATH, shift = 2000)  ``` 
 
 with the title:
 
 ``` {Dataset}_{Sample}_Shifted_{shift}_{epi_marker}_{gene_name}_{strand}_{diagnostic}.txt ```
 
 For example:
 

 ``` GSE67471_GSM1647620_Shifted_2000_H3K4me3_SGK1_neg_OG.txt```
 
 
 This function is found in the .py file Epigenetics_newgenes.py, which can be run as:
 
 
  ```nohup python3 Epigenetics_newgenes.py &```

**4) Store the position and signal values in numpy arrays:** 

Now we can use the function  ``` input_target_vectors(PATH)  ``` to store the Position histogram limiting bins and the corresponding Signal in numpy arrays. It also outputs the array Target with the target values (1/0 for Cancer gene: yes/no).

**5) Data preprocessing**

The data preprocessing cell crops the largest genes and zero pads the shortest up to a certain length (set to be 5.000 bp or 40.000 bp). The code also adds the missing values in the sequences by assigning to the signal gaps the previous signal value until a new one is found. This step is necessary since two independent experiments will show completely different bin limits, with different lengths, which need to be comparable. Zeroes are added two basepairs before and after the gene start and end to make sure that each track is properly defined.

The signals will be further binned at the first stages of the CNN, in order to reduce their dimensionality.


Finally, the data is reshaped into three dimensions (sequence_sample,basepairs,tracks) in order to be fed into the model, and also splitted into training,validation and test.

**6) The model**

THE ARCHITECTURE: Origene

We will approach the problem using a deep neural network, OriGENE, that will be constituted by the following elements:

*Inputs:* Multiple 1D aligned sequences (5.000/ 40.000 bp long) starting 2000 bp before the transcription start site (TSS) of the curated genes. We would expect this region to contain peaks of H3K4me3 related to promoters and devoided regions for enhancers. Healthy tissue and its matching cancer sample will be aligned and introduced as two different input tracks.Several histone modifier tracks will be also introduced in parallel to the network. The inputs will be binned using average pooling to reduce the dimensionality of the sequences while keeping the relevant structure.

*Feature extractor:* In order to extract noticeable but also hidden features, the first layers of the network will be convolutional layers (ReLU activation) and inception layers, aimed to create several filters. We will combine them with average and maxpooling layers that will allow us to reduce the dimensionality of the data while keeping the most significant values.

*Final classifier (Fully connected layers):*    
Finally, we will flatten all the filters and introduce the resulting vector to a multilayer perceptron (MLP, ReLU activation), a set of fully connected layers that will perform the classification.

*Output:* The last hidden layer will be connected to a single output node via a sigmoidal activation function. This will allow us to map 1: Cancer driver (CD) being Oncogene (OG) or Tumor supressor gene (TSG), 0: Neutral Gene (NG) in the target node.





In [ ]:
# Basic packages
import os
import numpy as np
import matplotlib.pyplot as plt
import random as rnd
import time
import re #package to split lines with different delimiters

# Tensorflow and Keras:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, Activation
from tensorflow.keras.layers import Conv1D, MaxPooling1D, AveragePooling1D, BatchNormalization, ZeroPadding1D, UpSampling1D
from tensorflow.keras.layers import TimeDistributed, Multiply
from tensorflow.keras.layers import Lambda, Concatenate, concatenate
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, RNN, Bidirectional, Reshape
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Nadam
from tensorflow.keras import backend as K
from keras.preprocessing.sequence import pad_sequences
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers

# Sklearn:
from sklearn import preprocessing
from sklearn.metrics import *
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# To have the plots inside the notebook "inlin" should be True. 
# If "inlin" = False, then plots will pop out of the notebook
inlin = True # True/False
if inlin:
    %matplotlib inline
else:
    %matplotlib


import matplotlib
# Uncomment the following lines to plot he figures with latex typography.
#matplotlib.use("pgf")
#matplotlib.rcParams.update({
#    "pgf.texsystem": "pdflatex",
#    'font.family': 'serif',
#    'text.usetex': True,
#   'pgf.rcfonts': False,
#})

# Extra libraries for signal processing
from contextlib import redirect_stdout
from scipy.signal import butter, lfilter, freqz, savgol_filter
from scipy.signal import find_peaks, resample
from scipy.interpolate import interp1d
from scipy import stats

**Choose patient to study**

*Patient options: string with the name* 

``` python
'Lung_I', 'Lung_II', 'Liver_I', 'Liver_II', 'NT2D1_I'
    
```

**Choose type of study**

``` python
'pan_cancer','tissue_specific'
    
```


**Choose epigenetic markers**

*Epigenetic marker track options: list with markers*

For lung and liver patients:

```python
['H3K4me3']
```

For NT2D1: any combination of the following markers.

```python
['H3K4me1','H3K4me3','H3K9me3','H3K9ac','H3K27me3','H3K36me3']
```



In [ ]:
patient = 'Liver_II'
study_type = 'tissue_specific'
track_list = ['H3K4me3']

### <center> 1.2 Read ChIP-seq output txt files and create target vector <center>
    

    


In [ ]:
tissue = patient.split('_')[0]
sample = patient.split('_')[1]

# cell_line_list: Tissue of interest
cell_line_list = [tissue]

if study_type == 'tissue_specific':
    main_PATH = '/scratch/lema/marc/oncogenes/GSE67471/Combined_txt_files_Liver_Specific_rmdup/'
    if sample == 'I':
        status_list = ['Normal'] #,'Cancer']
    else: # For patient II
        status_list = ['Normal_I'] #,'Cancer_I']
else:
    if tissue == 'NT2D1':
        main_PATH = '/scratch/lema/marc/oncogenes/GSE31755/Combined_txt_files_NT2D1_rmdup/'
    else:
        main_PATH = '/scratch/lema/marc/oncogenes/GSE67471/Combined_txt_files_All_Binary_rmdup/'
        if sample == 'I':
            status_list = ['Normal'] #,'Cancer']
        else: # For patient II
            status_list = ['Normal_I'] #,'Cancer_I']
            
print('Patient: {}, Tissue: {}, Sample: {}'.format(patient,tissue,sample))

**Secondary paths for the creation of the files:**

Once we count with bedgraph.sorted files, we want to do two things:

   - Split the files into different chromosomes in order to ease the search for the location of each gene:

   - Obtain the sequences corresponding to each gene (starting 2000 bp before the transcription start site) and assigning a target to the sequence, namely CD or NG.
   
To split the bedgraph.sorted files into single chromosomes, use the following command when being at the final/output directory:

   ``` bash
   awk  '{print > "{SRA file title}_"$1".txt"}' /scratch/lema/marc/oncogenes/Snakemake_workflow/results/05_bedgraph_sorted_files/SRR1947881.1.bedgraph.sorted 
   
   ```
    
To create separated files for each gene use Epigenetics_newgenes.py, which contains the function ``` create_txt ```, and run it as:

```
nohup python3 Epigenetics_newdata.py &
```

With the set of files required.


Note that the directories where we want to store the chromosomes, e.g. 
*'/scratch/lema/marc/oncogenes/GSE67471/chromosomes/GSM1647618_chr/'*

, and the directories where the final sequences for each gene will be stored, e.g. 

*'.../Combined_txt_files_Training_Validation_Binary/txt_files_Lung_Normal/txt_files_H3K4me3/'* 

must be created beforehand. 

<mark> Also, **please note that the code is meant to be run for files stored in directories that follow the scheme** .../txt_files_Tissue_Healthstatus/txt_files_Epimarker and will not work otherwise. <mark>






In [ ]:
def input_target_vectors(PATH):
    
    """ 
    This function creates lists containing the Numpy arrays (of varying length)
    for the location of the signal from a starting basepair to the end of the gene,
    the corresponding Signal values and the Target labels for each sample.
    
                                     __________ Inputs_____________
    PATH: Previous outPATH, where we created all the text files with the information for each gene.
                                      _________ Outputs ____________
    Position: list containing Numpy arrays of the position coordinates for each sample.
    Signal: list containing Numpy arrays with the signal values at each given position in the gene.
    Target: list with target labels for each sample.
    Genes: list of genes as they were loaded into the arrays
    """
    
    directory = [f.name for f in os.scandir(PATH)]
    directory = sorted(directory)
    Position = []
    Signal = []
    Target = []
    Genes = []

    for filename in directory: # For each document we will retreive the data for the position x and the signal value y.
        Genes.append(filename.split('_')[5]) #Adding new gene to gene list
        position = np.array([])
        signal = np.array([])
        with open(PATH+filename,'r') as inputFile:
            firstline = True #First line has a special status since we also want the first position coordinate
            for line in inputFile.readlines():
                if line[0] == 'c':
                    splitline = line.split()
                    if firstline == True:
                        lastpos = int(splitline[2])
                        first_basepair = np.float(splitline[1])
                        position = np.append(position,first_basepair)
                        signal = np.append(signal,np.float(splitline[3]))
                        position = np.append(position,np.float(splitline[2]))
                        firstline = False
         
                    else:
                        if int(splitline[1]) == lastpos:
                            lastpos = int(splitline[2])
                            position = np.append(position,np.float(splitline[2])) #-first_basepair)
                            signal = np.append(signal,np.float(splitline[3]))
                        else: # There is a gap in between two bins
                            #Filling the gap with zeroes
                            signal = np.append(signal,0.)
                            position = np.append(position,np.float(splitline[1])) #-first_basepair)
                            #Adding the new signal after the gap
                            signal = np.append(signal,np.float(splitline[3]))
                            position = np.append(position,np.float(splitline[2]))
                            lastpos = int(splitline[2])
                            
        # Second to last element in the file's name gives us the strand: pos/neg:
        split_name = filename.split('_')
        print('{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}'.format(split_name[0],
                                                  split_name[1],
                                                  split_name[3],
                                                  split_name[4],
                                                  split_name[5],
                                                  split_name[6],
                                                  split_name[7][:-4]), end='\r')
        
        # If the gene is encoded in the upper strand, we'll read it backwards,i.e.,
        # the signal is inverted. Position values are sparse, so the procedure will be:
        #1) the array must be substracted its maximum value to have a final position value of 0
        #2) absolute value of the array to have positive distances
        #3) flip the array in the reversed direction
             
        if split_name[-2] == 'neg':
            # Signal matched to end of the bin
            Position.append(np.flipud(abs(position[1:])))
            #Position.append(np.flipud(abs(position[1:]-position[-1])))  # Now the starting positionis the end of the last bin, therefore we skip the first bin
            Signal.append(np.flipud(signal))
            
        else:   
            # Signal matched to the beginning of the bin
            Position.append(position[:-1]) # The signal value will be assigned to the beginning of the step.
            Signal.append(signal)
        
        # The last part of the filename gives us the label: Cancer driver yes or no.

        if split_name[-1] == 'NG.txt': # Neutral genes will be assigned the target 0.
            Target = np.append(Target,[0.])
        else: # Oncogenes and Tumor-supressors will be assigned the target 1.
            Target = np.append(Target,[1.])
              
    return Position,Signal,Target,Genes

To know how many reads were mapped, we can use the command:
``` samtools view -c -F 4 SRR1947881.1.bam.sorted.rmdup
```

Duplicated reads (PCR induced mostly) were removed before counting the mapped reads.

In [ ]:
################## TEST CELL: Normalization values #######################################

if tissue == 'Lung' or tissue == 'Liver':
    # Duplicates removed and only mapped reads:
    # For the first dataset, GSE67471:
    Index = {'GSM1647618': {'Tissue':'Lung', 'Channel':'Normal', 'Reads': 34963546},
             'GSM1647619': {'Tissue':'Lung', 'Channel':'Cancer', 'Reads': 66048923},
             'GSM1647620': {'Tissue':'Liver','Channel':'Normal', 'Reads': 33963444},
             'GSM1647621': {'Tissue':'Liver','Channel':'Cancer', 'Reads': 47459130},
             'GSM1647622': {'Tissue':'Liver','Channel':'Normal_I', 'Reads': 29153715},
             'GSM1647623': {'Tissue':'Liver','Channel':'Cancer_I', 'Reads': 30034579},
             'GSM1647624': {'Tissue':'Lung', 'Channel':'Normal_I', 'Reads': 24715000},
             'GSM1647625': {'Tissue':'Lung', 'Channel':'Cancer_I', 'Reads': 28547876}}
else: 
    # For NT2D1:
    Index = {'GSM788071': {'Tissue': 'NT2D1', 'Channel': 'H3K27me3', 'Reads': 20525537},
             'GSM788072': {'Tissue': 'NT2D1', 'Channel': 'H3K4me3', 'Reads': 16123074},
             'GSM788080': {'Tissue': 'NT2D1', 'Channel': 'H3K9me3', 'Reads': 24386417},
             'GSM788081': {'Tissue': 'NT2D1', 'Channel': 'H3K36me3', 'Reads': 27546099},
             'GSM788083': {'Tissue': 'NT2D1', 'Channel': 'H3K4me1', 'Reads': 19772943},
             'GSM788086': {'Tissue': 'NT2D1', 'Channel': 'H3K9ac', 'Reads': 15673762}}



### <center> 3. Storing the position values, signal values and targets in arrays: <center>

In [ ]:
# Dictionaries where we'll store data and metadata of each track: filenames,position,signal and target values in an ordered way

Position_dict = {}
Signal_dict = {}
Target_dict = {}
Directory_dict = {}
Gene_dict = {}

print('Loading sequences:\n')
print('{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}\t{:10s}'.format('Dataset','Sample','Shift (bp)', 'Marker', 'Gene', 'Strand','Category'))
print('__________________________________________________________________________________________________________')


if tissue == 'Liver' or tissue == 'Lung': # Save in dictionary different tracks for different health status, not epi marker

    for status in status_list:
        Position_dict[status] = []
        Signal_dict[status] = []
        Directory_dict[status] = []

    Directory_dict_1 = {}
    Position_dict_1 = {}
    Signal_dict_1 = {}
    Target_dict_1 = {}
    
    for line in cell_line_list:
        for status in status_list:
            outPATH = main_PATH+'txt_files_{}_{}/txt_files_H3K4me3/'.format(line,status)
            Position_dict_1[status],Signal_dict_1[status],Target_dict[status], Gene_dict[status] = input_target_vectors(outPATH)
            Directory_dict_1[status] = sorted([f.name for f in os.scandir(outPATH)])

            Directory_dict[status] += Directory_dict_1[status]
            Position_dict[status] += Position_dict_1[status]
            Signal_dict[status] += Signal_dict_1[status]
            
else: # For NT2D1 we save the directory in terms of the tracks, not the status.
    for track in track_list:
        Position_dict[track] = []
        Signal_dict[track] = []
        Directory_dict[track] = []

    Directory_dict_1 = {}
    Position_dict_1 = {}
    Signal_dict_1 = {}
    Target_dict_1 = {}
    
    for line in cell_line_list:
        for track in track_list:
            outPATH = main_PATH+'txt_files_{}/txt_files_{}/'.format(line,track)
            Position_dict_1[track],Signal_dict_1[track],Target_dict[track], Gene_dict[track] = input_target_vectors(outPATH)
            Directory_dict_1[track] = sorted([f.name for f in os.scandir(outPATH)])

            Directory_dict[track] += Directory_dict_1[track]
            Position_dict[track] += Position_dict_1[track]
            Signal_dict[track] += Signal_dict_1[track]
    

### <center> 4. Plotting the raw signals <center> 
    
This test cell is aimed to illustrate how can the signals look like before preprocessing them. The length of the genes will change gene to gene. Since negative strand sequences are still not flipped, one can notice that the enrichment patterns for such sequences appear at the right hand side. One can also get an impression of the amount of noisy sequences

In [ ]:
################################## Plotting the resulting H3K4me3 profiles ###################
first_seq = 0
last_seq = len(Directory_dict[list(Directory_dict.keys())[0]])
step = 100
color_list = ['darkgreen','darkred','goldenrod','blue', 'orange','purple']
for element in range(first_seq, last_seq,step): #list_element:
    print('Sequence number:',element)
    channels = list(Gene_dict.keys())
    print(channels)
    gene = Gene_dict[channels[0]][element]
    print('Gene:', gene)
    plt.figure(figsize=(10,2.5))
    title = '{}_{}'.format(element,gene) 
    for i,channel in enumerate(channels):
        #print(Position_dict[track][element])
        plt.plot(Position_dict[channel][element],Signal_dict[channel][element], color= color_list[i],alpha=1,label='{}_{}'.format(tissue,channel), lw = .8)    

    plt.yticks(size=12)
    plt.ticklabel_format(style='sci', scilimits=(0,0))
    plt.legend(loc='upper left',
          ncol=1, fancybox=True, shadow=True, fontsize=12)
    plt.ylabel('Number of reads', fontsize=12)
    plt.xlabel('Basepair number in the chromosome', fontsize=12)
    #plt.savefig('Full_gene.png',dpi=200, bbox_inches='tight')
    plt.show()

            

# <center> SECTION II: DATA PREPROCESSING <center> 
    
In this section the data will be preprocessed and standarized.




    
The most basic information regarding our dataset is obtained in a separate notebook: Dataset_information.ipynb.
There one can find typical gene lengths, chromosome and strand distributions for the three categories of genes.
    
In the next cell, we will create the training and test sets, with input and target, from the Signal array that was created before.
        
### <center> 2.1 DATA PREPROCESSING <center> 

1) Alignment: Alignment of the different tracks is crucial if we want to make the most of the shared spatial distribution and the convolutional layers. Since all the different tracks for the same gene start at different points (depending on the Chip-Seq experiment reads), we will allign the markers and zero pad the beginning and end of the sequences that start after -2000bp from the TSS and end before the desired length.

    
2) Standardization to a given sequencing depth. The amount of DNA used for each sample, the amount enzyme aimed to cut the DNA into fragments (Micrococcal Nuclease, MNase) that is used, the different PCR cycles, etc. are factors that will contribute to have a different number of reads (50 first basepairs of each found and sequenced fragment) for each of the samples. Since these reads will be then piled up and will give us an "enrichment intensity equivalent", which are our final signals, we will need to bring all samples to a common standard. The standard was set to 30.000 reads, more or less the number of unduplicated and mapped reads for the different samples in GSE67471.
    


In [ ]:
##################################### PARAMETERS ###########################################################:
step_width = 1   # FIXED.
n_features = 1   # Number of input channels: we will merge different tracks afterwards.
normalization_reads = 30000000 # Standard total number of reads to which we'll map our signals
#normalization_reads = 2000000

#basepair_number = 5000 # Length of the cropped and zero-padded final signals
basepair_number = 40000
len_threshold = basepair_number//step_width # Maximum length for the sequences in case of zero pad. Max baspair/width of the step

In [ ]:
Uniform_Position = np.arange(0.,len_threshold*step_width,step_width) # Position index
xtrn_dict = {}
Normalize = True
min_max_start_strand = [] # list containing position of basepair origin for each sequence + strand
channel_list = list(Position_dict.keys())

for track in channel_list:
    # Loading the Position and Signal arrays for a single track.
    Position = Position_dict[track]

    #Normalize sequencing depth:
    if Normalize == False:
        Signal = Signal_dict[track]
    else:
        # Finding corresponding original sample read number:
        for element in Index.values():
            if element['Tissue'] == cell_line_list[0] and element['Channel'] == track:
                reads= element['Reads']
                print(element['Tissue'],element['Channel'], 'multiplication factor:',normalization_reads/reads)
        Signal = list(np.array(Signal_dict[track])*normalization_reads/reads)

    xtrn = [] # Imputed Signal array to be filled

    for i,element in enumerate(Position):
        # Printing the evolution of the loading process
        frac_20 = int(i/len(Position)*20)
        print('Appending sequence {}/{}: [{}{}{}]'.format(i, len(Position),frac_20*'=','>',(20-frac_20-1)*' '), end='\r')
        ############################# ALIGNMENT OF THE TRACKS:#####################################3
        Uniform_Signal = []
        posindex = 1
        pos = element[0]
        # Reversed sequences
        if element[0] > element[1]:
            min_start = element[0]
            max_start = element[0]
            for track_II in channel_list:
                first_value = Position_dict[track_II][i][0]
                if first_value < min_start:
                    min_start = Position_dict[track_II][i][0]
                elif first_value >= max_start:
                    max_start = Position_dict[track_II][i][0]
                    
            for k in range(int((max_start-min_start)-(element[0]-min_start))): # Filling with zeroes before the start
                Uniform_Signal.append([0., ] )
            while (pos > (max_start-basepair_number)):
                if pos <= element[posindex-1] and pos > element[posindex]:
                    Uniform_Signal.append([Signal[i][posindex-1], ] )
                    pos -= step_width
                elif pos <= element[-1]: # Zero padding scenario
                    Uniform_Signal.append([0., ] )
                    pos -= step_width
                else:
                    posindex +=1
            # Cropping the end of the genes that are not the one starting at the lowest basepair in negative strand.
            #Uniform_Signal = np.array(Uniform_Signal[int(element[0]-min_start):])
            min_max_start_strand.append(['-',min_start])
        # Positive strand sequences:
        else: 
            max_start = element[0]
            min_start = element[0]
            for track_III in channel_list:
                first_value = Position_dict[track_III][i][0]
                if first_value > max_start:
                    max_start = Position_dict[track_III][i][0]
                elif first_value <= min_start:
                    min_start = Position_dict[track_III][i][0]
            for k in range(int((max_start-min_start)-(max_start-element[0]))): # Filling with zeroes before the start
                Uniform_Signal.append([0., ] )
            while (pos < (min_start+basepair_number)): 
                if pos >= element[posindex-1] and pos < element[posindex]:
                    Uniform_Signal.append([Signal[i][posindex-1], ] )
                    pos += step_width
                elif pos >= element[-1]: # Zero padding scenario
                    Uniform_Signal.append([0., ] )
                    pos += step_width
                else:
                    posindex +=1
            # Cropping the beggining of the genes that are not the one starting at the lowest basepair in negative strand.
            #Uniform_Signal = np.array(Uniform_Signal[int(max_start-element[0]):])
            min_max_start_strand.append(['+',max_start])

        xtrn.append(Uniform_Signal)

    xtrn = np.array(xtrn)
    xtrn_dict[track] = np.array(xtrn)

    #####################################################################################################################



        
# Reshape from [samples, timesteps] into [samples, timesteps, features] for the input to the LSTM layer:
for track in list(Position_dict.keys()):
    xtrn_dict[track] = xtrn_dict[track].reshape((xtrn_dict[track].shape[0], xtrn_dict[track].shape[1], n_features))

# Final arrays
xtrn = np.concatenate(tuple(xtrn_dict.values()), axis = 2) # Merging all tracks into a single input matrix, for both inputs:
ytrn = np.array(Target_dict[channel_list[0]]) #Redefined target list

        
print('\nTarget vector (ytrn): \n', ytrn)
print("\nDataset shape:",xtrn.shape)
print('\nFirst values of the final xtrn:\n',xtrn[0,:10,:])


#Saving starting basepair, sequences and corresponding targets in npy array files:

```python


final_seq_file = 'Final_sequences_40kbp_{}'.format(patient)
final_target_file = 'Final_targets_40kbp_{}'.format(patient)

np.save(final_seq_file, xtrn, allow_pickle=True, fix_imports=True)
np.save(final_target_file, ytrn, allow_pickle=True, fix_imports=True)


#basepair_origin_file = 'Basepair_origin_file_{}_{}'.format(cell_line_list[0],track_list[0])
#np.save(basepair_origin_file, min_max_start_strand, allow_pickle=True, fix_imports=True)
```

### <center> 2.2 FINAL PLOTS <center> 

The plots of the final sequences show the H3K4 trimethylation pattern starting around 2000 bp before the TSS of the gene with a value for every single basepair. They do not start exactly 2000 bp before given the cropping system explained in the previous section, necessary to align different samples with each other. Furthermore, the algorithm just requires to find the pattern somewhere in the signal, given the translational invariance of CNNs.


In [ ]:
####################### Plotting the final cropped and resampled input signals: ####################################
colorlist = ['darkgreen', 'darkred', 'goldenrod','blue','orange','purple']

first_seq = 0
last_seq = len(Directory_dict[list(Directory_dict.keys())[0]])
max_bp = 40000
step = 10

for element in range(first_seq,last_seq,step):  
    channels = list(Gene_dict.keys())
    gene = Gene_dict[channels[0]][element]
    label = Target_dict[channels[0]][element]
    plt.figure(figsize=(10,2.5))
    title = 'Sequence number: {}\nGene name: {}\nTarget category: {}\n'.format(element,gene,label) 
    print(title)
    for i,track in enumerate(channel_list):
        plt.plot(np.arange(len(xtrn[element,:max_bp,i])),xtrn[element,:max_bp,i], color= colorlist[i],alpha=1, lw = .8, label = track)
    plt.xlabel('Basepair number',fontsize=20)
    plt.ylabel('Frequency',fontsize=20)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    #plt.xlim((0,maxpos))
    plt.legend(fontsize=12)
    #plt.savefig('/home/william/marc/Pictures/Original_signal.png', dpi=100, bbox_inches='tight')
    plt.show()




### <center> 2.3 ESTIMATING THE PERCENTAGE OF NOISY SEQUENCES<center> 
    
In an ideal case scenario, noise could be assessed and removed by comparing all H3K4me3 tracks with an Input track, a no-antibody control, which was not obtained by immunoprecipitating only the fragments bound to our targetted marker. This track would allow us to set a background number of counts. Since we did not count with such track for GSE67471, the threshold for "low enrichment" became a rather arbitrary choice.

In [ ]:
# Percentage of noisy sequences

signal_threshold = 5
print('Signal_threshold:', signal_threshold)
CD_counter = 0
NG_counter = 0
CDs = 0
NGs = 0

for element in range(len(Position)):
    target = ytrn[element]
    max_val = 0
    for i,track in enumerate(channel_list):
        track_max = max(xtrn[element,:,i])
        if  track_max > max_val:
            max_val = track_max
    # Counting cancer driver genes:
    if int(target) == 1:
        CDs += 1
        if max_val < signal_threshold:
            CD_counter += 1
    # Counting neutral genes
    else:
        NGs += 1
        if max_val < signal_threshold:
            NG_counter += 1

signal_values = xtrn.flatten()
binsize=1

plt.figure(figsize=(4,4))
plt.hist(signal_values, bins=np.arange(2,50,binsize))
plt.xlabel('Signal intensity (Normalized reads)')
plt.ylabel('Signal value frequency')
plt.show()


print('CDs: {}, NGs: {}'.format(CDs,NGs))        
print('CD Counter:{}, Percentage of noise signals: {}'.format(CD_counter, CD_counter/CDs))
print('NG Counter:{}, Percentage of noise signals: {}'.format(NG_counter, NG_counter/NGs))



### <center> 2.4 SIGNAL CLUSTERING <center>
    
In this section we will assess the potential of a PCA to separate signals according to different categories: healthy vs. cancer tissue, CD vs. NG sequence, for the Liver and Lung samples.
    
    

In [ ]:
# Importing the required packages
import seaborn as sns
import pandas as pd
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
savepath = '/home/william/marc/marc_directory/OriGENE_paper/Images/' #Path where the images will be saved

angle_list = 30*np.sin(np.arange(0,2*np.pi,.03))+45
angleI_list = 30*np.cos(np.arange(0,2*np.pi,.03))+45
counter = 0

if tissue == 'Liver' or tissue == 'Lung':

    for angle,angleI in ([45,50],[45,50]):#zip(angle_list,angleI_list):

        plt.figure(figsize=(15,3))
        grid = plt.GridSpec(3, 12, wspace=4., hspace=2.)

        cols = ['bp={:d}'.format(int(i)) for i in Uniform_Position] # Columns
        # Healthy tissue dataframe
        df_normal = pd.DataFrame(xtrn[:,:,0].reshape((xtrn[:,:,0].shape[0],xtrn[:,:,0].shape[1])), columns = cols) 
        # Cancer tissue dataframe
        df_cancer = pd.DataFrame(xtrn[:,:,1].reshape((xtrn[:,:,1].shape[0],xtrn[:,:,1].shape[1])), columns = cols)


        ################# Principal component analysis (PCA) for healthy tissue ########################
        pca_normal = PCA(n_components=10)
        pca_normal_result = pca_normal.fit_transform(df_normal[cols].values)

        df_normal['pca 1'] = pca_normal_result[:,0]
        df_normal['pca 2'] = pca_normal_result[:,1]
        df_normal['pca 3'] = pca_normal_result[:,2]

        pc_df = pd.DataFrame(data = pca_normal_result , 
                columns = ['1', '2','3','4','5','6','7','8','9','10'])
        df = pd.DataFrame({'Eigenvalues':pca_normal.explained_variance_,
                     'PC':['1','2','3','4','5','6','7','8','9','10']})

        plt.subplot(grid[:, :4])
        sns.barplot(x='PC',y="Eigenvalues", 
                   data=df, label='Healthy tissue', alpha=.75,linewidth=1, facecolor='darkgreen',
                         edgecolor="darkgreen")
        #plt.title('Scree plot normal healthy tissue')
        #plt.yticks(ticks = [5000,10000,15000,20000])
        plt.xlabel('Component number')
        plt.legend()
        #plt.savefig('Healthy_scree.png')

        # Cumulated variance:
        cum_var_1 = np.cumsum(np.round(pca_normal.explained_variance_ratio_,decimals=3)*100)


        ################# Principal component analysis (PCA) for cancer tissue ########################

        pca_cancer = PCA(n_components=10)
        pca_cancer_result = pca_cancer.fit_transform(df_cancer[cols].values)

        df_cancer['pca 1'] = pca_cancer_result[:,0]
        df_cancer['pca 2'] = pca_cancer_result[:,1]
        df_cancer['pca 3'] = pca_cancer_result[:,2]

        pc_df1 = pd.DataFrame(data = pca_cancer_result , 
                columns = ['1', '2','3','4','5','6','7','8','9','10'])
        df1 = pd.DataFrame({'Eigenvalues':pca_cancer.explained_variance_,
                     'PC':['1','2','3','4','5','6','7','8','9','10']})

        plt.subplot(grid[:, 4:8])
        sns.barplot(x='PC',y="Eigenvalues", 
                   data=df1, label='Cancer tissue', alpha=.75,linewidth=1, facecolor='darkred',
                         edgecolor="darkred");
        #plt.title('Scree plot cancer tissue')
        plt.xlabel('Component number')
       # plt.yticks(ticks = [5000,10000,15000,20000], labels=['','','',''])
        #plt.ticklabel_format(style='sci', scilimits=(0,0))
        plt.legend()
        #plt.savefig('Cancer_scree.png')

        # Cumulated variance for cancer tissue:
        cum_var_2 = np.cumsum(np.round(pca_cancer.explained_variance_ratio_,decimals=3)*100)



        plt.subplot(grid[:, 8:12])
        #plt.title('PCA VARIANCE EXPLAINED')
        plt.ylabel('\% of variance explained')
        plt.xlabel('Component number')
        plt.ylim(0,100.5)
        plt.xlim(1,10)
        plt.axhline(y=80, color='silver', linestyle='--')
        plt.plot(np.arange(1,11,1),cum_var_1,color= 'darkgreen' , label='Healthy tissue', alpha=1.)
        plt.plot(np.arange(1,11,1),cum_var_2,color= 'darkred', label='Cancer tissue', alpha=1.)
        plt.savefig(savepath+'Healthy_vs_cancer_{}.png'.format(patient), dpi = 500, bbox_inches='tight')
        plt.legend()
        plt.show()



        ################################ Writing everything in a file: ########################################


        with open('Scree_plot_summary.txt', 'a') as f1:
            f1.write('########{} {} {}  #######\n'.format(main_PATH.split('/')[-2].split('_')[-2],cell_line_list[0],track_list[0]))
            f1.write('Healthy tissue Scree \n')
            f1.write(str(pca_normal.explained_variance_))
            f1.write('\n Healthy tissue cumulated variance \n')
            f1.write(str(cum_var_1))
            f1.write('\n \n')

            f1.write('Cancer tissue Scree \n')
            f1.write(str(pca_cancer.explained_variance_))
            f1.write('\n Cancer tissue cumulated variance \n')
            f1.write(str(cum_var_2))
            f1.write('\n \n \n')

        #########################################################################


        # Second figure:

        fig = plt.figure(figsize=(15,7))
        ax = fig.add_subplot(1, 3, 1, projection='3d')
        ax.grid(False)
        ax.view_init(angleI , angle)
        # Data for three-dimensional scattered points
        ax.scatter3D(df_normal['pca 1'],df_normal['pca 2'],df_normal['pca 3'], alpha = .4, color = 'darkgreen', label='Healthy', marker='.', lw=0.1)
        ax.scatter3D(df_cancer['pca 1'],df_cancer['pca 2'],df_cancer['pca 3'], alpha = .4, color = 'darkred', label='Tumor', marker = '.', lw=0.1)
        #plt.title(" PCA space representation")
        plt.xlabel(' PC 1')
        plt.ylabel(' PC 2')
        ax.set_zlabel(' PC  3')
        plt.ticklabel_format(style='sci', scilimits=(0,0))
        #ax.set_xticklabels([])
        #ax.set_yticklabels([])
        #ax.set_zticklabels([])
        #plt.legend(loc= 'lower center')
        #plt.savefig(savepath+'Fig{}.png'.format(angle), dpi = 500, bbox_inches='tight')




        #################################     Clustering      ###################################################


        def plot_dendrogram(model, **kwargs):
            # Create linkage matrix and then plot the dendrogram

            # create the counts of samples under each node
            counts = np.zeros(model.children_.shape[0])
            n_samples = len(model.labels_)
            for i, merge in enumerate(model.children_):
                current_count = 0
                for child_idx in merge:
                    if child_idx < n_samples:
                        current_count += 1  # leaf node
                    else:
                        current_count += counts[child_idx - n_samples]
                counts[i] = current_count

            linkage_matrix = np.column_stack([model.children_, model.distances_,
                                              counts]).astype(float)

            # Plot the corresponding dendrogram
            dendrogram(linkage_matrix, **kwargs)

            return linkage_matrix


        linkage_list = ['ward', 'average', 'complete', 'single']
        cluster = AgglomerativeClustering(n_clusters=2, affinity='euclidean', linkage=linkage_list[0])
        cluster.fit_predict(df_normal)

        # CLUSTERING RESULTS

        ax = fig.add_subplot(1, 3, 2, projection='3d')
        ax.view_init(angleI, angle )
        ax.grid(False)
        ax.scatter3D(df_normal['pca 1'].values[np.where(cluster.labels_ == 1)],df_normal['pca 2'].values[np.where(cluster.labels_ == 1)],df_normal['pca 3'].values[np.where(cluster.labels_ == 1)], c= 'darkorange', marker='.', lw=0.1, label='Cluster 1' )
        ax.scatter3D(df_normal['pca 1'].values[np.where(cluster.labels_ != 1)],df_normal['pca 2'].values[np.where(cluster.labels_ != 1)],df_normal['pca 3'].values[np.where(cluster.labels_ != 1)], c= 'darkblue', marker='.', lw=0.1, label='Cluster 2' )


        #plt.title(" PCA space representation clustering")
        #plt.xlabel(' PC 1')
        #plt.ylabel(' PC 2')
        #ax.set_zlabel(' PC 3')
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        #plt.ticklabel_format(style='sci', scilimits=(0,0))
        #plt.legend(loc= 'lower center')
        #plt.savefig(savepath+'Fig_clust{}.png'.format(angle), dpi = 500, bbox_inches='tight')


        # GROUND TRUTH LABELS

        ax = fig.add_subplot(1, 3, 3, projection='3d')
        #get current axes
        ax.view_init(angleI, angle)
        ax.grid(False)
        # Data for three-dimensional scattered points
        ax.scatter3D(df_normal['pca 1'].values[np.where((Target_dict[channel_list[0]]) == 1.)],df_normal['pca 2'].values[np.where((Target_dict[channel_list[0]]) == 1.)],df_normal['pca 3'].values[np.where((Target_dict[channel_list[0]]) == 1.)], c= 'darkmagenta', marker='.', lw=0.1, label='CD')
        ax.scatter3D(df_normal['pca 1'].values[np.where((Target_dict[channel_list[0]]) != 1.)],df_normal['pca 2'].values[np.where((Target_dict[channel_list[0]]) != 1.)],df_normal['pca 3'].values[np.where((Target_dict[channel_list[0]]) != 1.)], c= 'darkcyan', marker='.', lw=0.1, label='NG')
        #plt.title(" PCA space representation clustering true label")
        #plt.xlabel(' PC 1')
        #plt.ylabel(' PC 2')
        #ax.set_zlabel(' PC  3')
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        #plt.legend(loc= 'lower center')
        #plt.ticklabel_format(style='sci', scilimits=(0,0))
        plt.subplots_adjust(wspace=0.05, hspace=0.05)
        plt.savefig(savepath+'{}_{}_3D-results.png'.format(patient,counter), dpi=200, bbox_inches='tight')
        plt.show()

        ##### Plot pca components (eigenvectors)
        plt.figure(figsize=(25,5))
        for i in range(5):
            plt.plot(np.arange(len(pca_normal.components_[0]))[:],pca_normal.components_[i][:], label = 'PC {}'.format(i+1))
        plt.legend()
        plt.show()

        plt.figure(figsize=(25,5))
        for i in range(10):
            plt.plot(np.arange(len(pca_cancer.components_[0]))[:],pca_cancer.components_[i][:], label = 'PC {}'.format(i+1))
        plt.legend()
        plt.show()
        counter +=1


# <center> SECTION III: TRAINING <center> 

In [ ]:
#Saving the ordered set of sequences for future prediction
xtrn_CD_prediction = xtrn
ytrn_CD_prediction = ytrn

channels = list(Gene_dict.keys())
gtrn_CD_prediction = np.array(Gene_dict[channels[0]])


### <center> 3.2 Auxiliary visualization functions <center>

In [ ]:
def binary_pred_stats(ytrue, ypred, threshold=0.5):
    """ Function that, from the true labels and the predicted model outputs, gives us
        the main performance metrics. Mattias Ohlsson's code"""
    one_correct = np.sum((ytrue==1)*(ypred > threshold))
    zero_correct = np.sum((ytrue==0)*(ypred <= threshold))
    sensitivity = one_correct / np.sum(ytrue==1)
    specificity = zero_correct / np.sum(ytrue==0)
    accuracy = (one_correct + zero_correct) / len(ytrue)
    precision = one_correct / np.sum(ypred > threshold)
    return sensitivity, specificity, accuracy, precision

def plot_confusion_matrix(cm,
                          target_names,
                          cell_line,
                          sample,
                          savepath,
                          fold,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    
    """ Function that plots the confusion matrix given cm. Mattias Ohlsson's code extended."""

    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(4,3))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    #plt.title(title)
    plt.colorbar()
    #plt.ylim([-0.5, cm.shape[0]-0.5])

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=0, fontsize=12)
        plt.yticks(tick_marks, target_names,fontsize=12)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black", fontsize=14)
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black", fontsize=14)


    plt.tight_layout()
    plt.ylabel('True label', fontsize=14)
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass), fontsize=14)
    plt.savefig(savepath+'{}_{}_Fold_{}_Confusion.png'.format(cell_line,sample,fold), dpi=50, bbox_inches='tight')
    plt.show()
    
def layerVisualization(model,
                     xtest, 
                     ytest,
                     idx,
                     savepath,
                     layer_type = Conv1D ):
    """ This function is aimed to visualize the outputs of all the filters given a certain type of layer.
    Based on Mattias Ohlsson's code"""
    
    
    outs = [l.output for l in model.layers if isinstance(l, layer_type)]
    intermediate = K.function([model.layers[0].input], outs)
    print(ytest[idx])
    states = [xtest[idx:idx+1]] + intermediate([xtest[idx:idx+1]])
    print(states[0])
    plt.figure(figsize=(18,12)) 
    output_flow = []
    for k,layer in enumerate(states):
        filters = []
        for ifilter in range(np.shape(layer[0])[1]):
            filters.append(layer[0][:,ifilter])
        output_flow.append(filters)
    return output_flow

### <center> 3.3 THE MODEL <center>
    
In this section the OriGENE model is presented.

### INCEPTION BLOCKS 

The inception module introduces convolutions at different scales that are stacked at the end of the layer. It reduces the number of trainable parameters by using kernel 1 convolutions before operating further, and it allowed us to obtain a complex enough feature extractor while keeping the size of the model rather low.

**ADDITIONAL INFORMATION**
  
INCEPTION links of interest: 

https://www.analyticsvidhya.com/blog/2018/10/understanding-inception-network-from-scratch/ 
https://machinelearningmastery.com/how-to-implement-major-architecture-innovations-for-convolutional-neural-networks/

In [ ]:
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, BatchNormalization, Activation, Add, Flatten, Dense)
from tensorflow.keras.models import Model


# Function for creating a projected inception module with kernel 1 convolutions that reduce significantly the 
# number of trainable parameters
def inception_module(layer_in, f1, f2_in, f2_out, f3_in, f3_out, f4_out):
    """
    Inception module from:
    https://machinelearningmastery.com/how-to-implement-major-architecture-innovations-for-convolutional-neural-networks/
    Adapted to a 1D problem. Kernel regularizers have been added in order not to overfit to training data.
    """
    
    reg_value = 1e-2
    # 1x1 conv
    conv1 = Conv1D(f1, kernel_size = 1, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(layer_in)
    # 3x3 conv
    conv3 = Conv1D(f2_in, kernel_size = 1, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(layer_in)
    conv3 = Conv1D(f2_out, kernel_size = 3, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(conv3)
    # 5x5 conv
    conv5 = Conv1D(f3_in, kernel_size = 1, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(layer_in)
    conv5 = Conv1D(f3_out, kernel_size = 5, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(conv5)
    # 3x3 max pooling
    pool = MaxPooling1D( pool_size= 3, strides= 1, padding='same')(layer_in)
    pool = Conv1D(f4_out, kernel_size= 1, padding='same', activation='relu', kernel_regularizer = regularizers.l2(reg_value))(pool)
    # concatenate filters, assumes filters/channels last
    layer_out = concatenate([conv1, conv3, conv5, pool], axis=-1)
    return layer_out



### Architectures

In [ ]:
def Model_OriGENE(Input_shape):
    """
    ################################################################################################################
                                            The OriGENE final Model
    ################################################################################################################
    
    Description:
    
        OriGENE consists of I) an Input section, II) a Feature extractor with convolutional layers and
        III) a classifier.
    
        I) The model takes as inputs 40kbp aligned sequences with the local enrichment level of a certain epigenetic 
        marker. Longer genes were cropped while shorter genes have been zero-padded to the desired length in 
        the preprocessing steps.
        
        Examples: H3K4me3 for Healthy-Tumour matching tissue, xtrn.shape = (None,40000,2)
                  H3K4me1, H3K4me3, H3K9ac, H3K9me3, H3K27me3, H3K6me3 aligned tracks, xtrn.shape = (None,40000,6)
                  

        II) From these set of sequences associated to a gene, the network extracts relevant features to
        classify the gene as Cancer Driver (CD) or Neutral Gene (NG). Inception modules help to ease the learning
        process, extracting information at varying spatial scales while reducing the number of trainable parameters.
        L2 regularization is applied in the convolutional layers.
        
        III) The classifier integrates the extracted features into a unified response: the probability of the given 
        set of sequences to describe a CD or a NG. High dropout rates help to avoid overfitting.
    
    _______________________________________________________________________________________________________________
    Inputs. Input_shape: Last two indices of the training array's shape, i.e. (None,40000,2) -> (40000,2)
    
    Outputs. model: OriGENE model.
    
    _______________________________________________________________________________________________________________
    Usage: See cell below.
    
    """
    kernel_size = 3
    signal = Input(shape=Input_shape, dtype=np.float32, name='signal')
    x = signal
    x = AveragePooling1D(pool_size= 20)(x)
    x = Conv1D(4, kernel_size, activation='relu',strides=2, kernel_regularizer=regularizers.l2(1e-2))(x)
    x = Conv1D(8, kernel_size, activation='relu',strides=2, kernel_regularizer=regularizers.l2(1e-2))(x)
    #x = Conv1D(16, kernel_size, activation='relu', kernel_regularizer=regularizers.l2(1e-2))(x)
    x = AveragePooling1D(pool_size= 2)(x)
    # INCEPTION
    x = inception_module(x, f1=2, f2_in=2, f2_out=2, f3_in=2, f3_out=2, f4_out=2)
    x = inception_module(x, f1=2, f2_in=2, f2_out=2, f3_in=2, f3_out=2, f4_out=2)
    x = MaxPooling1D(pool_size= 2)(x)
    x = inception_module(x, f1=4, f2_in=4, f2_out=4, f3_in=4, f3_out=4, f4_out=4)
    x = inception_module(x, f1=4, f2_in=4, f2_out=4, f3_in=4, f3_out=4, f4_out=4)
    x = MaxPooling1D(pool_size= 2)(x)
    #
    x = Flatten()(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)
    diagn = Dense(1, activation='sigmoid')(x)
    model = Model(signal, diagn)
    adam = Adam(lr=0.0004)
    model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])
    print(model.summary())
    return model




### <center> 3.4 Model performance assessment:  K-fold cross testing of the model <center>
    

In [ ]:
########################## Hyperparameters ###############################################
n_splits = 5 # 5-fold cross testing
skf = StratifiedKFold(n_splits= n_splits , shuffle=True, random_state=4) # Original state was 4 # Stratified K-fold parameters
fold = 1 # Fold counter initalization
show_sequences = False #Plot the sequences corresponding to the validation set
show_filters = True # Plot the outputs of the intermediate filters
savepath = '/home/william/marc/marc_directory/OriGENE_paper/Code/Model_evaluation/' #Path where the images will be saved
batchsize = 53 #22 #53 # 97 # Size of the minibatches
nE = 100       # The number of epochs

Avg_performance_metrics = {'Sensitivity': 0 , 'Specificity': 0 , 'Accuracy': 0 , 'Precision': 0}
Avg_AUC = 0
#########################################################################################

y_true_global = np.array([])
y_pred_global = np.array([])
genes_global = np.array([])

for train_index, val_index in skf.split(xtrn_CD_prediction,ytrn_CD_prediction):
    
    
    # We split the data 10 times: 9 parts for training and the remaining for validation in a stratified manner
    xtrn, xval = xtrn_CD_prediction[train_index], xtrn_CD_prediction[val_index]
    gtrn, gval = gtrn_CD_prediction[train_index], gtrn_CD_prediction[val_index]
    ytrn, yval = ytrn_CD_prediction[train_index], ytrn_CD_prediction[val_index]

    # Information about the dataset and the current fold:
    print('Fold:', fold)
    print('Indexes for the training sequences;',len(train_index), train_index)
    print('Indexes for the validation sequences', len(val_index), val_index)
    print('Training labels:',ytrn)
    print('Training set shape:',np.shape(xtrn))
    print('---------------')

    #Defining the model:
    Input_shape = xtrn.shape[1:]     
    model = Model_OriGENE(Input_shape)
    
    # Save the weights in an hdf5 file and fit the model to the available data:
    model_checkpoint = ModelCheckpoint(savepath+'OriGENE_{}_40kbp_singletrack_{}.hdf5'.format(patient,fold), monitor='loss',verbose=1, save_best_only=True)
    estimator_model = model.fit([xtrn], ytrn, validation_data=([xval], yval), epochs=nE, shuffle = False, batch_size=batchsize, callbacks=[model_checkpoint])    
    
    # Training history
    print(estimator_model.history.keys())
    plt.figure(figsize=(4,4))
    plt.ylabel('$Loss$ $/$ $Accuracy$', size= 14)
    plt.xlabel('$Epoch$', size= 14)
    plt.plot(estimator_model.history['loss'], label = 'Loss', color = 'darkgreen')
    plt.plot(estimator_model.history['val_loss'], label = 'Validation loss', color = 'lightgreen')
    plt.plot(estimator_model.history['accuracy'], label = 'Accuracy', color = 'darkblue')
    plt.plot(estimator_model.history['val_accuracy'], label = 'Validation accuracy', color = 'lightblue')
    plt.legend(loc='best', fontsize=14)
    #plt.ylim([0.4,2])
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    #plt.savefig(savepath+'OriGENE_Final_Metrics_{}_fold_{}.png'.format(patient,fold))
    plt.show()
    
    # Loss and accuracy evolution in two axis:
    plt.figure(figsize=(4,4))
    ax1 = plt.subplot(111)
    l1, = ax1.plot(estimator_model.history['loss'], label = 'Loss', color = 'darkgreen')
    l11, = ax1.plot(estimator_model.history['val_loss'], label = 'Validation loss', color = 'lightgreen')
    ax1.set_ylabel('Loss', size=14)
    ax1.set_xlabel('Epochs', size=14)
    ax2 = ax1.twinx()
    ax2.set_ylim((0,1))
    ax2.set_ylabel('Accuracy', size=14)
    l2, = ax2.plot(estimator_model.history['accuracy'], label = 'Accuracy', color = 'darkblue')
    l22, = ax2.plot(estimator_model.history['val_accuracy'], label = 'Validation accuracy', color = 'lightblue')
    plt.legend([l1, l11, l2, l22], ['Loss', 'Validation Loss', 'Accuracy', 'Validation Accuracy'])
    plt.xlim([0,100])
    ax1.tick_params(axis="x", labelsize=14)
    ax1.tick_params(axis="y", labelsize=14)
    ax2.tick_params(axis="y", labelsize=14)
    plt.savefig(savepath+'OriGENE_Final_Metrics_{}_fold_{}.png'.format(patient,fold))
    plt.show()
        
    # ROC curve and AUC:
    #y_pred = model.predict([xval,vval]).flatten()
    y_pred = model.predict([xval]).flatten()
    y_true = yval.flatten()
    
    y_pred_global = np.append(y_pred_global, np.array(y_pred))
    y_true_global = np.append(y_true_global, np.array(y_true))
    genes_global = np.append(genes_global, np.array(gval))
    
    fpr,tpr,thresholds = roc_curve(y_true, y_pred)
    sensitivity = tpr
    specificity = 1-fpr

    roc_auc = auc(fpr, tpr)
    Avg_AUC += roc_auc/n_splits

    plt.figure(figsize = (4,4))
    lw = 2
    plt.plot(fpr, tpr, color='darkorange',lw=lw, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic example')
    plt.legend(loc="lower right")
    plt.savefig(savepath+'OriGENE_{}_Fold_{}_ROC.png'.format(patient,fold))
    plt.show()
    


    # Get the training predictions and results for those
    #sens, spec, acc = binary_pred_stats(ytrn_CIG_prediction.flatten(), predtrain)
    sens, spec, acc, prec = binary_pred_stats(y_true, y_pred)
    Avg_performance_metrics['Sensitivity'] += sens/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Specificity'] += spec/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Accuracy'] += acc/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Precision'] += prec/n_splits # Add the contribution of this fold
    
    print('training: Accuracy = {:.4f}, Sensitivity = {:.4f}, Specificity = {:.4f}, Precision = {:.4f}'.format(acc, sens, spec, prec), '\n')

    ################################## Confusion matrix ##################################################

    y_pred_bin = np.array([round(x) for x in y_pred])
    confusion = confusion_matrix(y_true, y_pred_bin)

    target_names = ['NG','CD']
    plot_confusion_matrix(cm           = confusion,
                          cell_line    = cell_line_list[0],
                          sample       = 'OriGENE',
                          savepath     = savepath,
                          fold         = fold,
                          target_names = target_names,
                          normalize    = False,
                          title        = "Confusion Matrix")
       
    ############################################# Sequences and their classification ###########################
    
    # Plotting the validation sequences while printing their predicted values and their target assignation in different
    if show_sequences == True:
        y_pred = model.predict(xval)
        for i,element in enumerate(val_index):
            print('Sequence:',element,'Gene:',gtrn_CD_prediction[element], 'Predicted:',y_pred[i], ytrn_CD_prediction[element])
            #(len(Position)):  
            plt.figure(figsize=(20,3))
            title = ''
            for k,track in enumerate(channel_list):
                title += ' '+str(element)+Directory_dict[track][element]+'\n'
                plt.plot(Uniform_Position,xtrn_CD_prediction[element,:,k], color= (rnd.uniform(0,1),rnd.uniform(0,1),rnd.uniform(0,1)),alpha=1, lw = .8, label = track)    

            plt.title(title)
            plt.xlabel('basepair number from the beginning of the gene',fontsize=15)
            plt.ylabel('Frequency',fontsize=15)
            plt.legend()
            plt.show()
            
    ######################################## Intermediate filter outputs ###########################################3
        
    # Plotting the outputs of the filters corresponding to the specified layer type:
    if show_filters == True:
        for seq_idx in range(len(val_index)):
            if val_index[seq_idx] == 1060:
                print('Sequence index:{}'.format(val_index[seq_idx]))
                output_flow = layerVisualization(model, xval,yval, idx = seq_idx,savepath='/home/william/marc/Pictures/', layer_type = Conv1D) # Layer types: AveragePooling1D, MaxPooling1D, Conv1D
    fold +=1
    


In [ ]:
# Plotting flow of outputs from convolutional filters

import matplotlib.gridspec as gridspec

color_list = ['darkgreen','darkred']
label_list = ['Healthy tissue', 'Cancer tissue']

if show_filters == True:

    plt.figure(figsize = (13,15))
    grid = plt.GridSpec(40, 5, wspace=.1, hspace=0.5 )

    grid_lims = [0,4,9,12,15,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39]
    print(len(grid_lims))

    for i,layer in enumerate(output_flow):
        if i == 0:
            ax1 = plt.subplot(grid[grid_lims[i]:grid_lims[i+1],:4])
            plt.axis('off')
            j=0
            for channel in layer:
                plt.plot(np.arange(0,len(channel)),channel, color = color_list[j], label = label_list[j], lw = .7 )
                plt.legend()
                #plt.xlim((0,40000))
                j += 1
        else:
            ax1 = plt.subplot(grid[grid_lims[i]:grid_lims[i+1],:])
            plt.axis('off')
            plt.imshow(layer, cmap='seismic', aspect='auto')
            plt.colorbar(ticks=[np.min(layer),np.max(layer)], shrink=.8, aspect =5)
            #plt.xlim((0,int(np.shape(layer)[1])/2))
            #print(np.shape(layer))
    plt.savefig('Filter_visualization_5kbp.png'.format(i)) 
    plt.show()

    grid_lims = [0,2,4,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33]

    plt.figure(figsize = (8,40))
    grid = plt.GridSpec(33, 1, wspace=.1, hspace=1. )
    for i,layer in enumerate(output_flow):
        ax1 = plt.subplot(grid[grid_lims[i]:grid_lims[i+1],:])
        for j,filtr in enumerate(layer):
            plt.plot(np.arange(int(len(filtr)/1)),filtr[:int(len(filtr)/1)], lw =.4, label='{},{}'.format(i,j))
            plt.xlim((0,int(len(filtr)/1)))
            if j == 0:
                plt.ylabel('Input')
            else:
                plt.ylabel('Activation')
            #plt.legend()
    plt.xlabel('Node')
    plt.savefig('Filter_visualization_lines_5kbp.png')
    plt.show() 

In [ ]:
# Printing the weights of the first layers:

Input_shape = xtrn.shape[1:]    
#model = Model_INCEPTION_original(Input_shape)
model = Model_OriGENE(Input_shape)
model.load_weights(savepath+"OriGENE_Lung_I_fold_1.hdf5")
weights = np.array(model.get_weights())

#print('Weights[0]:',weights[0].shape)
#print(weights[0])
#print('Biases:',weights[1])


for filtr in range(weights[0].shape[2]):
    plt.figure(figsize=(3,2))
    plt.imshow(np.rot90(weights[0][:,:,filtr], k=3), cmap='bwr',vmin=-0.2,vmax=0.2)
    #plt.imshow(weights[0][:,:,filtr], cmap='bwr')
    plt.axis('off')
    plt.colorbar(shrink = .8)
    plt.text(1.1, -.7, 'Weights from healthy track to filter {}'.format(filtr+1), ha='center', fontsize=8)
    plt.text(1.1, 1.7, 'Weights from cancer track to filter {}'.format(filtr+1), ha='center', fontsize=8)
    plt.show()
    


#Weights:

#The weights must be specified in a three-dimensional structure, in terms of rows, columns, and channels, for each layer.
# In a Conv layer, first index gives [0] weights and [1] corresponding biases. Second index gives original source node 
#(position in filter) and final index corresponds to each filter.

# In a dense layer, first index gives [0]
# weights and [1] corresponding biases. Second index gives source node from the original layer and last index gives target node
# at the subsequent layer.

# In bias, the second index is directly the last index, target node.

##########################################################################################################################
weights = np.array(model.get_weights())
print('Weights[0]:',weights[0].shape)
print(weights[0])
print('Biases:',weights[1])


Max = 0.
Min = 0. 
for element in weights:
    elmax = np.max(element)
    elmin = np.min(element)
    if elmax > Max:
        Max = elmax
    elif elmin < Min:
        Min = elmin
            
print('Minimum weight:', Min)
print('Maximum weight:', Max) 

In [ ]:
plt.figure(figsize = (50,5))
plt.title('Weights')
layer = 50
grid = plt.GridSpec(1,weights[layer].shape[2], wspace=.1, hspace=0.5 )

for filtr in range(weights[layer].shape[2]):
    ax1 = plt.subplot(grid[:,filtr])
    plt.imshow(np.rot90(weights[layer][:,:,filtr], k=3), cmap='bwr', vmin = -0.25,vmax = 0.25)
    #plt.imshow(weights[0][:,:,filtr], cmap='bwr')
    plt.axis('off')
    #plt.colorbar(shrink = .8)
    #plt.text(1.1, -.7, 'Weights from healthy track to filter {}'.format(filtr+1), ha='center', fontsize=8)
    #plt.text(1.1, 1.7, 'Weights from cancer track to filter {}'.format(filtr+1), ha='center', fontsize=8)
plt.colorbar()
plt.savefig('Weights.png')
plt.show()

**Saving all predictions for different folds and their true labels together:**

In [ ]:
#``` python

################################### Saving y_pred_global and y_true_global #############################
final_y_pred = 'OriGENE_{}_specific_40kbp_singletrack__ypred'.format(patient,patient)
final_y_true = 'OriGENE_{}_specific_40kbp_singletrack_ytrue'.format(patient,patient)
final_genes  = 'OriGENE_{}_specific_40kbp_singletrack_genes'.format(patient,patient)

np.save(final_y_pred, y_pred_global, allow_pickle=True, fix_imports=True)
np.save(final_y_true, y_true_global, allow_pickle=True, fix_imports=True)
np.save(final_genes , genes_global , allow_pickle=True, fix_imports=True)

    
######################################3 Writing the final results in a file ############################3
Summary_file = 'Summary_file_OriGENE.txt'

with open(savepath+Summary_file,'a') as file:
    file.write('################ Rmdup ##############################')
    file.write('Patient: {} 40kbp singletrack  \n'.format(patient,patient))
    file.write('Final averaged metrics:\n Acc:{}\t Sp:{}\t Sn:{}\t Prec:{}\n'
               .format(Avg_performance_metrics['Accuracy'],
                       Avg_performance_metrics['Specificity'],
                       Avg_performance_metrics['Sensitivity'],
                       Avg_performance_metrics['Precision']))
    file.write('Averaged AUC:\t {}\n'.format(Avg_AUC))
    file.write('---------------------------------\n')

#```

###  <center>  Gene category prediction: rankings <center>

In [ ]:
# Writing rankings:

PATH = '/home/william/marc/marc_directory/OriGENE_paper/Code/Model_evaluation/'

with open(PATH+'Ranking_{}_H3K4me3_5kbp.txt'.format(patient),'w') as outputFile:
    #Now we can print the most important positive predicted sequences:
    outputFile.write('________________ CD PROBABILITY RANKING _______________ \n')
    outputFile.write('Sequence: \t Probability \n')
    #Prediction:

    y_pred = np.load('OriGENE_{}_H3K4me3_5kbp_ypred.npy'.format(patient))
    y_true = np.load('OriGENE_{}_H3K4me3_5kbp_ytrue.npy'.format(patient))
    genes = np.load('OriGENE_{}_H3K4me3_5kbp_genes.npy'.format(patient))
    
    Dictionary = {}
    
    for i,gene in enumerate(genes):
        Dictionary[gene+'\t'+str(y_true[i])] = y_pred[i]

    #Sorting the values of the dictionary:
    sorted_Dictionary = sorted(Dictionary.items(), key=lambda x:x[1], reverse=True)

    for element in sorted_Dictionary[:100]:
        outputFile.write('{}\t{}\n'.format(element[0],element[1]))



###  <center>  Cross-patient prediction <center>

In [ ]:

fold = 1

n_splits = 5 # 5-fold cross testing
skf = StratifiedKFold(n_splits= n_splits , shuffle=True, random_state=4) # Original state was 4 # Stratified K-fold parameters
savepath = '/home/william/marc/marc_directory/OriGENE_paper/Code/Model_evaluation/' #Path where the images will be saved

#########################################################################################
Avg_performance_metrics = {'Sensitivity': 0 , 'Specificity': 0 , 'Accuracy': 0 , 'Precision': 0}
Avg_AUC = 0
y_true_global = np.array([])
y_pred_global = np.array([])
genes_global = np.array([])



for train_index, val_index in skf.split(xtrn_CD_prediction,ytrn_CD_prediction):
    
    xval =  xtrn_CD_prediction[val_index]
    gval =  gtrn_CD_prediction[val_index]
    yval =  ytrn_CD_prediction[val_index]
    
    Input_shape = xtrn.shape[1:]    
    #model = Model_INCEPTION_original(Input_shape)
    model = Model_OriGENE(Input_shape)
    model.load_weights(savepath+"OriGENE_Lung_I_cross_tissue_5kbp_fold_{}.hdf5".format(fold))
    
    y_pred = model.predict([xval]).flatten()
    y_true = yval.flatten()
    
    y_pred_global = np.append(y_pred_global, np.array(y_pred))
    y_true_global = np.append(y_true_global, np.array(y_true))
    genes_global = np.append(genes_global, np.array(gval))
    
    # Get the training predictions and results for those
    #sens, spec, acc = binary_pred_stats(ytrn_CIG_prediction.flatten(), predtrain)
    sens, spec, acc, prec = binary_pred_stats(y_true, y_pred)
    Avg_performance_metrics['Sensitivity'] += sens/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Specificity'] += spec/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Accuracy'] += acc/n_splits # Add the contribution of this fold
    Avg_performance_metrics['Precision'] += prec/n_splits # Add the contribution of this fold
    
    fold += 1
    

################################### Saving y_pred_global and y_true_global #############################
final_y_pred = 'OriGENE_Lung_I_cross_tissue_5kbp_predicting_{}_ypred'.format(patient)
final_y_true = 'OriGENE_Lung_I_cross_tissue_5kbp_predicting_{}_ytrue'.format(patient)
final_genes  = 'OriGENE_Lung_I_cross_tissue_5kbp_predicting_{}_genes'.format(patient)

np.save(final_y_pred, y_pred_global, allow_pickle=True, fix_imports=True)
np.save(final_y_true, y_true_global, allow_pickle=True, fix_imports=True)
np.save(final_genes , genes_global , allow_pickle=True, fix_imports=True)

    
######################################3 Writing the final results in a file ############################3
Summary_file = 'Summary_file_OriGENE.txt'

with open(savepath+Summary_file,'a') as file:
    file.write('################ Rmdup ##############################')
    file.write('Patient:  Lung_I_cross_tissue_5kbp predicting {}  \n'.format(patient))
    file.write('Final averaged metrics:\n Acc:{}\t Sp:{}\t Sn:{}\t Prec:{}\n'
               .format(Avg_performance_metrics['Accuracy'],
                       Avg_performance_metrics['Specificity'],
                       Avg_performance_metrics['Sensitivity'],
                       Avg_performance_metrics['Precision']))
    file.write('Averaged AUC:\t {}\n'.format(Avg_AUC))
    file.write('---------------------------------\n')